# Convert B1Z1 PACT-Pos weights into a coupled PACT hot start

This notebook mirrors `model_checkpoint_work.ipynb` for the current B1Z1 models. It copies trained PACT-Pos model parameters into a freshly constructed coupled PACT model and writes a weights-only checkpoint. Optimizer state, PPO iteration, KL controller state, force-gate state, and other training state are deliberately excluded.

In [6]:
from pathlib import Path
import sys
import torch

# Locate the repository whether Jupyter starts in the repo root or this folder.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "rsl_rl").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "rsl_rl").is_dir():
    raise RuntimeError("Run this notebook from inside HCR_Genesis_PACT_Development")
sys.path.insert(0, str(repo_root))

from rsl_rl.modules.actor_critic_b1z1_pact import (
    ActorCriticB1Z1PACT, B1Z1PACTDecoder,
)
from rsl_rl.modules.actor_critic_b1z1_pact_pos import ActorCriticB1Z1PACTPos

## Paths

Set `source_checkpoint_path` to a trained B1Z1 PACT-Pos checkpoint. The output path is repository-relative by default and can be changed freely.

In [7]:
source_checkpoint_path = repo_root / "logs/b1z1/b1z1_pact_pos_gym/Aug17_19-26-28_b1z1_pact_pos/model_10000.pt"
output_checkpoint_path = (
    repo_root
    / "rsl_rl/modules/pretrained_checkpoints/b1z1_pact/pact_pos_hotstart.pt"
)

print("Source:", source_checkpoint_path)
print("Output:", output_checkpoint_path)

Source: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/logs/b1z1/b1z1_pact_pos_gym/Aug17_19-26-28_b1z1_pact_pos/model_10000.pt
Output: /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/rsl_rl/modules/pretrained_checkpoints/b1z1_pact/pact_pos_hotstart.pt


## Construct matching source and target models

These values mirror the current `B1Z1PACTPosCfgPPO` and `B1Z1PACTCfgPPO` definitions. Keeping them visible makes architecture drift fail clearly during strict checkpoint loading.

In [8]:
NUM_ACTOR_OBS = 81
NUM_CRITIC_OBS = 419 * 5
NUM_ACTIONS = 17
HISTORY_DIM = 81 * 25
LATENT_DIM = 64
EXPLICIT_DIM = 23
PRIVILEGED_RECON_DIM = 232

POSITION_INIT_STD = [0.80, 1.00, 1.00] * 4 + [0.85] * 5
POSITION_MIN_STD = [0.15, 0.25, 0.25] * 4 + [0.15] * 5

common_actor_kwargs = dict(
    num_actor_obs=NUM_ACTOR_OBS,
    num_critic_obs=NUM_CRITIC_OBS,
    num_actions=NUM_ACTIONS,
    history_dim=HISTORY_DIM,
    latent_dim=LATENT_DIM,
    actor_layers=(512, 256, 128),
    critic_layers=(1024, 512, 256, 128),
    context_layers=(512, 256, 128),
    explicit_decoder_layers=(128, 64),
    explicit_dim=EXPLICIT_DIM,
    film_hidden_dim=64,
    activation="elu",
    max_noise_std=1.1,
)

pact_pos_actor = ActorCriticB1Z1PACTPos(
    **common_actor_kwargs,
    init_noise_std=POSITION_INIT_STD,
    min_noise_std=POSITION_MIN_STD,
)
pact_actor = ActorCriticB1Z1PACT(
    **common_actor_kwargs,
    init_noise_std=POSITION_INIT_STD * 2,
    min_noise_std=POSITION_MIN_STD * 2,
)
pact_pos_decoder = B1Z1PACTDecoder(
    LATENT_DIM, PRIVILEGED_RECON_DIM, hidden=(128, 256, 512), activation="elu"
)
pact_decoder = B1Z1PACTDecoder(
    LATENT_DIM, PRIVILEGED_RECON_DIM, hidden=(128, 256, 512), activation="elu"
)

## Load the PACT-Pos checkpoint

The source checkpoint may contain optimizer and training state, but only its two model state dictionaries are read.

In [9]:
source_checkpoint = torch.load(
    source_checkpoint_path, map_location="cpu", weights_only=True
)
required_keys = {"model_state_dict", "privileged_decoder_state_dict"}
missing_keys = required_keys.difference(source_checkpoint)
if missing_keys:
    raise KeyError(f"Source checkpoint is missing: {sorted(missing_keys)}")

def remove_compile_prefix(state_dict):
    prefix = "_orig_mod."
    return {
        (key[len(prefix):] if key.startswith(prefix) else key): value
        for key, value in state_dict.items()
    }

source_actor_state = remove_compile_prefix(source_checkpoint["model_state_dict"])
source_decoder_state = remove_compile_prefix(
    source_checkpoint["privileged_decoder_state_dict"]
)
pact_pos_actor.load_state_dict(source_actor_state, strict=True)
pact_pos_decoder.load_state_dict(source_decoder_state, strict=True)
print("Strict PACT-Pos source load passed.")

Strict PACT-Pos source load passed.


## Transfer PACT-Pos into coupled PACT

All equal-shaped actor tensors are copied, including the trained torque-clone head. PACT-Pos has a 17-element Gaussian standard deviation while coupled PACT has 34 elements, so the trained values are copied into the position-action half; the torque-action half retains its configured PACT initialization. The decoder architectures are identical and copy strictly.

In [10]:
source_state = pact_pos_actor.state_dict()
target_state = pact_actor.state_dict()
copied, adapted, retained = [], [], []

for key, target_value in target_state.items():
    source_value = source_state.get(key)
    if source_value is None:
        retained.append((key, "missing from PACT-Pos"))
        continue
    if source_value.shape == target_value.shape:
        target_state[key] = source_value.detach().clone()
        copied.append(key)
        continue
    if key == "std" and target_value.numel() == 2 * source_value.numel():
        converted = target_value.detach().clone()
        converted[:source_value.numel()] = source_value.detach()
        converted[source_value.numel():] = source_value.detach() * 0.6
        target_state[key] = converted
        adapted.append((key, tuple(source_value.shape), tuple(target_value.shape)))
        continue
    retained.append((key, f"shape {tuple(source_value.shape)} -> {tuple(target_value.shape)}"))

pact_actor.load_state_dict(target_state, strict=True)
pact_decoder.load_state_dict(pact_pos_decoder.state_dict(), strict=True)

print(f"Copied actor tensors: {len(copied)}")
print("Adapted actor tensors:", adapted)
print("Retained fresh PACT tensors:", retained)

Copied actor tensors: 42
Adapted actor tensors: [('std', (17,), (34,))]
Retained fresh PACT tensors: [('_std_clip_lwr', 'shape (17,) -> (34,)'), ('_std_clip_upr', 'shape (17,) -> (34,)')]


## Save and verify the weights-only checkpoint

The output contains exactly two keys expected by `B1Z1PACTRunner._load_pretrained_model()`. Normal training resume checkpoints are separate and still include optimizer/training state.

In [11]:
weights_only_checkpoint = {
    "model_state_dict": pact_actor.state_dict(),
    "privileged_decoder_state_dict": pact_decoder.state_dict(),
}
assert set(weights_only_checkpoint) == {
    "model_state_dict", "privileged_decoder_state_dict"
}

output_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(weights_only_checkpoint, output_checkpoint_path)

# Strictly reload into fresh target modules before declaring success.
verification_checkpoint = torch.load(
    output_checkpoint_path, map_location="cpu", weights_only=True
)
verification_actor = ActorCriticB1Z1PACT(
    **common_actor_kwargs,
    init_noise_std=POSITION_INIT_STD * 2,
    min_noise_std=POSITION_MIN_STD * 2,
)
verification_decoder = B1Z1PACTDecoder(
    LATENT_DIM, PRIVILEGED_RECON_DIM, hidden=(128, 256, 512), activation="elu"
)
verification_actor.load_state_dict(verification_checkpoint["model_state_dict"], strict=True)
verification_decoder.load_state_dict(
    verification_checkpoint["privileged_decoder_state_dict"], strict=True
)

print(f"Saved verified weights-only hot start to {output_checkpoint_path}")
print("Configure B1Z1PACTCfgPPO.policy.pretrained_path with this path to use it.")

Saved verified weights-only hot start to /home/oyoungquist/Research/Genesis_Development/HCR_Genesis_PACT_Development/rsl_rl/modules/pretrained_checkpoints/b1z1_pact/pact_pos_hotstart.pt
Configure B1Z1PACTCfgPPO.policy.pretrained_path with this path to use it.
